# 01. 서울 전월세 데이터 파이프라인

**목적:** 2024년 서울 25개 자치구의 아파트 전월세 거래량을 중심으로 인구이동, 기준금리, 생활인구 데이터를 결합하여 머신러닝용 월별 데이터셋을 생성합니다.

실행 순서: **환경 설정 → 국토교통부 API 수집 → KOSIS 전처리 → ECOS 전처리 → 서울 생활인구 전처리 → MySQL 저장/결합 → ML 테이블 생성**

> 이 노트북은 프로젝트 루트의 `notebooks/` 폴더에서 실행하는 것을 기준으로 작성되었습니다. `.env`와 원본 CSV 파일을 먼저 준비해야 합니다.

## 0. 사전 준비

필요 파일:
- `../.env` : MOLIT API Key 및 MySQL 접속정보
- `../data/raw/kosis_migration_2024.csv`
- `../data/raw/ecos_base_rate_2024.csv`
- `../data/raw/LOCAL_PEOPLE_GU_2024.csv`

`.env` 예시:
```text
MOLIT_API_KEY=
DB_HOST=localhost
DB_PORT=3306
DB_USER=root
DB_PASSWORD=
DB_NAME=seoul_rent
```

MySQL에서 먼저 `CREATE DATABASE seoul_rent;`를 실행합니다.

In [ ]:
# 공통 라이브러리와 환경변수 로드
import os
import time
import requests
import pandas as pd
import xml.etree.ElementTree as ET
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv("../.env", override=True)

API_KEY = os.getenv("MOLIT_API_KEY")
engine = create_engine(
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)

if not API_KEY:
    raise ValueError("MOLIT_API_KEY가 .env에 없습니다.")

with engine.connect() as conn:
    print("MySQL 연결 확인:", conn.execute(text("SELECT 1")).fetchone())

## 1. 국토교통부 아파트 전월세 실거래 수집

- 출처: 공공데이터포털 「국토교통부 아파트 전월세 실거래가 자료」
- API: `RTMSDataSvcAptRent/getRTMSDataSvcAptRent`
- 범위: 서울 25개 자치구, 2024년 1~12월
- 핵심 값: 보증금, 월세, 면적, 층, 건축연도, 계약일자

공공데이터포털에서 받은 **URL-encoded serviceKey**는 `requests`의 `params`에 다시 인코딩하지 않고 URL에 직접 연결합니다.

In [ ]:
# 서울 25개 자치구 법정동 코드
seoul_gu = {
    "종로구":"11110", "중구":"11140", "용산구":"11170", "성동구":"11200", "광진구":"11215",
    "동대문구":"11230", "중랑구":"11260", "성북구":"11290", "강북구":"11305", "도봉구":"11320",
    "노원구":"11350", "은평구":"11380", "서대문구":"11410", "마포구":"11440", "양천구":"11470",
    "강서구":"11500", "구로구":"11530", "금천구":"11545", "영등포구":"11560", "동작구":"11590",
    "관악구":"11620", "서초구":"11650", "강남구":"11680", "송파구":"11710", "강동구":"11740"
}

BASE_URL = "https://apis.data.go.kr/1613000/RTMSDataSvcAptRent/getRTMSDataSvcAptRent"

In [ ]:
# API 호출 함수: 자치구/월 단위로 XML 데이터를 DataFrame으로 반환
def fetch_rent(gu_name, gu_code, ymd, num_rows=5000):
    url = (
        f"{BASE_URL}?serviceKey={API_KEY}"
        f"&LAWD_CD={gu_code}&DEAL_YMD={ymd}"
        f"&numOfRows={num_rows}&pageNo=1"
    )
    response = requests.get(url, timeout=30)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    result_code = root.findtext(".//resultCode")
    if result_code not in (None, "000", "00"):
        raise RuntimeError(f"API 오류 {gu_name} {ymd}: {result_code} {root.findtext('.//resultMsg')}")

    total_count = int(root.findtext(".//totalCount") or 0)
    if total_count > num_rows:
        raise RuntimeError(f"{gu_name} {ymd}: {total_count}건으로 num_rows={num_rows} 초과")

    rows = []
    for item in root.findall(".//item"):
        row = {child.tag: child.text for child in item}
        row["guName"] = gu_name
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# 서울 전체 2024년 데이터 수집
all_frames = []
for gu_name, gu_code in seoul_gu.items():
    for month in range(1, 13):
        ymd = f"2024{month:02d}"
        temp = fetch_rent(gu_name, gu_code, ymd)
        all_frames.append(temp)
        print(gu_name, ymd, len(temp))
        time.sleep(0.2)

rent_raw = pd.concat(all_frames, ignore_index=True)
print("수집 완료:", rent_raw.shape)

In [ ]:
# 원본 보존
os.makedirs("../data/raw", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)
rent_raw.to_csv("../data/raw/seoul_rent_2024.csv", index=False, encoding="utf-8-sig")

### 1-1. 전월세 데이터 전처리
보증금의 쉼표 제거, 수치형 변환, 계약일 생성, 전세/월세 구분을 수행합니다.

In [ ]:
rent = rent_raw.copy()

rent["deposit"] = rent["deposit"].str.replace(",", "", regex=False).astype(int)
rent["monthlyRent"] = pd.to_numeric(rent["monthlyRent"], errors="coerce")

for col in ["excluUseAr", "floor", "buildYear", "dealYear", "dealMonth", "dealDay"]:
    rent[col] = pd.to_numeric(rent[col], errors="coerce")

rent["dealDate"] = pd.to_datetime(
    rent["dealYear"].astype(str) + "-" +
    rent["dealMonth"].astype(str) + "-" +
    rent["dealDay"].astype(str)
)
rent["rentType"] = rent["monthlyRent"].apply(lambda x: "전세" if x == 0 else "월세")

print(rent.shape)
print(rent["rentType"].value_counts())
print("주요 결측치:
", rent[["deposit","monthlyRent","dealDate","guName"]].isna().sum())

In [ ]:
# 정제 CSV 및 MySQL 저장
rent.to_csv("../data/processed/seoul_rent_2024_clean.csv", index=False, encoding="utf-8-sig")
rent.to_sql("rental_transactions", engine, if_exists="replace", index=False, chunksize=5000)
print("rental_transactions:", len(rent))

## 2. KOSIS 인구이동 데이터

- 출처: KOSIS 국가통계포털, 국내인구이동통계 (`DT_1B26001_A01`)
- 범위: 서울 25개 자치구, 2024년 월별
- 사용 변수: 총전입, 총전출, 순이동

KOSIS에서 내려받은 CSV는 2행 헤더 구조이므로 `header=[0, 1]`로 읽습니다.

In [ ]:
kosis = pd.read_csv("../data/raw/kosis_migration_2024.csv", encoding="utf-8", header=[0, 1])

rows = []
for _, row in kosis.iterrows():
    gu = row.iloc[0]
    for month in range(1, 13):
        ym = f"2024.{month:02d}"
        rows.append({
            "guName": gu,
            "yearMonth": ym.replace(".", "-"),
            "moveIn": row[(ym, "총전입 (명)")],
            "moveOut": row[(ym, "총전출 (명)")]
        })

migration = pd.DataFrame(rows)
migration["moveIn"] = pd.to_numeric(migration["moveIn"], errors="coerce")
migration["moveOut"] = pd.to_numeric(migration["moveOut"], errors="coerce")
migration["netMove"] = migration["moveIn"] - migration["moveOut"]

migration.to_csv("../data/processed/kosis_migration_2024_clean.csv", index=False, encoding="utf-8-sig")
migration.to_sql("migration", engine, if_exists="replace", index=False)
print("migration:", migration.shape)

## 3. 한국은행 ECOS 기준금리

- 출처: 한국은행 경제통계시스템(ECOS)
- 통계표: `1.3.1. 한국은행 기준금리 및 여수신금리`
- 범위: 2024년 1~12월
- 사용 변수: 한국은행 기준금리

In [ ]:
ecos = pd.read_csv("../data/raw/ecos_base_rate_2024.csv", encoding="utf-8")
rate_row = ecos[ecos["계정항목"] == "한국은행 기준금리"].iloc[0]

interest_rate = pd.DataFrame([
    {"yearMonth": f"2024-{month:02d}", "baseRate": rate_row[f"2024/{month:02d}"]}
    for month in range(1, 13)
])
interest_rate["baseRate"] = pd.to_numeric(interest_rate["baseRate"], errors="coerce")
interest_rate.to_sql("interest_rate", engine, if_exists="replace", index=False)
print("interest_rate:", interest_rate.shape)

## 4. 서울 열린데이터광장 생활인구

- 출처: 서울 열린데이터광장 「자치구 단위 서울 생활인구(내국인)」
- 원본: `LOCAL_PEOPLE_GU_2024.csv`
- 사용 변수: `tot_lvpop_co`
- 집계: 자치구 × 월별 평균 생활인구

In [ ]:
seoul_pop = pd.read_csv("../data/raw/LOCAL_PEOPLE_GU_2024.csv", encoding="utf-8")

# 기준일(YYYYMMDD) → YYYY-MM
s = seoul_pop["stdr_de_id"].astype(str).str[:6]
seoul_pop["yearMonth"] = s.str[:4] + "-" + s.str[4:6]

# 생활인구의 자치구 코드는 MOLIT의 자치구 코드와 대응
code_to_gu = {int(code): name for name, code in seoul_gu.items()}
seoul_pop["guName"] = seoul_pop["adstrd_code_se"].map(code_to_gu)

if seoul_pop["guName"].isna().any():
    raise ValueError("매핑되지 않은 자치구 코드가 있습니다.")

living_pop = (
    seoul_pop.groupby(["guName", "yearMonth"])["tot_lvpop_co"]
    .mean().reset_index()
    .rename(columns={"tot_lvpop_co": "avgLivingPop"})
)
living_pop.to_sql("living_population", engine, if_exists="replace", index=False)
print("living_population:", living_pop.shape)

## 5. MySQL 집계 및 데이터 결합

거래 원본을 `자치구 × 월` 단위 거래건수로 집계한 뒤 인구이동, 기준금리, 생활인구를 JOIN합니다. 최종 `ml_rent_market`은 머신러닝 노트북의 입력 테이블입니다.

In [ ]:
# 1) 전월세 거래량 + 인구이동
query = """
SELECT
    r.guName, r.yearMonth, r.contractCount,
    m.moveIn, m.moveOut, m.netMove
FROM (
    SELECT
        guName,
        CONCAT(dealYear, '-', LPAD(dealMonth, 2, '0')) AS yearMonth,
        COUNT(*) AS contractCount
    FROM rental_transactions
    GROUP BY guName, dealYear, dealMonth
) r
JOIN migration m
  ON r.guName = m.guName AND r.yearMonth = m.yearMonth
ORDER BY r.guName, r.yearMonth
"""
monthly_market = pd.read_sql(query, engine)
monthly_market.to_sql("monthly_rent_market", engine, if_exists="replace", index=False)
print("monthly_rent_market:", monthly_market.shape)

In [ ]:
# 2) 기준금리 결합
query = """
SELECT m.*, i.baseRate
FROM monthly_rent_market m
JOIN interest_rate i ON m.yearMonth = i.yearMonth
ORDER BY m.guName, m.yearMonth
"""
market_final = pd.read_sql(query, engine)
market_final.to_sql("monthly_rent_market_final", engine, if_exists="replace", index=False)
print("monthly_rent_market_final:", market_final.shape)

In [ ]:
# 3) 생활인구 결합 → 최종 ML 테이블
query = """
SELECT
    m.guName, m.yearMonth, m.contractCount,
    m.moveIn, m.moveOut, m.netMove, m.baseRate,
    l.avgLivingPop
FROM monthly_rent_market_final m
JOIN living_population l
  ON m.guName = l.guName AND m.yearMonth = l.yearMonth
ORDER BY m.guName, m.yearMonth
"""
ml_data = pd.read_sql(query, engine)
ml_data.to_sql("ml_rent_market", engine, if_exists="replace", index=False)
print("ml_rent_market:", ml_data.shape)
ml_data.head()

## 6. 최종 검증

2024년 25개 자치구 × 12개월이 모두 결합되었다면 최종 행 수는 **300행**입니다. 결측치와 중복 키도 함께 확인합니다.

In [ ]:
print("shape:", ml_data.shape)
print("자치구 수:", ml_data["guName"].nunique())
print("월 수:", ml_data["yearMonth"].nunique())
print("결측치:
", ml_data.isna().sum())
print("중복 guName-yearMonth:", ml_data.duplicated(["guName", "yearMonth"]).sum())

assert len(ml_data) == 300, "예상 행 수(300)와 다릅니다. 원본 데이터/결합 결과를 확인하세요."
assert ml_data.duplicated(["guName", "yearMonth"]).sum() == 0

## 산출물

- MySQL `rental_transactions`: 전월세 거래 원본 정제 테이블
- MySQL `migration`: 월별 인구이동
- MySQL `interest_rate`: 월별 기준금리
- MySQL `living_population`: 월별 평균 생활인구
- MySQL `ml_rent_market`: **최종 머신러닝 입력 데이터**

다음 노트북: `02_machine_learning.ipynb`